# Update cellsnp output dir 

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    list(
        
        library(Seurat), 
        
        library(ggplot2), 
        library(patchwork), 
        library(RColorBrewer), 
        
        library(tidyverse), 
        library(data.table), # fread
        
        library(reticulate)
        
    )
    
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

python:         /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/bin/python
libpython:      /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/libpython3.10.so
pythonhome:     /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT:/nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT
version:        3.10.16 | packaged by conda-forge | (main, Apr  8 2025, 20:53:32) [GCC 13.3.0]
numpy:          /nobackup/peer/fdeckert/miniconda3/envs/p.3.10.16-FD20250213HSCT/lib/python3.10/site-packages/numpy
numpy_version:  2.2.6

NOTE: Python version was forced by use_python() function

In [4]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [5]:
# Source files
source("plotting_global.R")
source("bin/seurat_qc.R")

In [6]:
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Load Seurat data 

In [7]:
# Store data as h5ad 
ad <- import("anndata")

adata <- ad$read_h5ad("data/object/qc.h5ad")

cnt <- t(adata$X)
colnames(cnt) <- adata$obs_names$to_list()
rownames(cnt) <- adata$var_names$to_list()

so <- Seurat::CreateSeuratObject(counts=cnt, meta.data=adata$obs)
so <- Seurat::AddMetaData(so, read.csv("result/annotation/annotation.csv", row.names=1))

In [8]:
so$celltype_domain <- gsub('[$\\\\]+', "", so$celltype_domain)

# Load vireo output

In [9]:
# Get reference cell_id ident
cell_id_ident <- data.frame(
    
    sample_name=so$sample_name, 
    cell_id_ident=sapply(strsplit(colnames(so), "_"), "[", 2)

)

rownames(cell_id_ident) <- NULL
cell_id_ident <- dplyr::distinct(cell_id_ident)

In [10]:
vireo_dir <- grep("Patient_", list.dirs("data/genotype/3_vireo/", recursive=FALSE), value=TRUE)

In [11]:
vireo_import <- function(vireo_dir, cell_id_ident) {
    
    vireo_df <- data.table::fread(paste0(vireo_dir, "/donor_ids.tsv"))
    
    # Extract meta data from cell column
    vireo_df$cell_id <- sapply(strsplit(vireo_df$cell, "_Baseline|_Tx|_D14|_D100|_aGVHD"), "[", 2)
    vireo_df$patient_id <- sapply(strsplit(vireo_df$cell, "_Baseline|_Tx|_D14|_D100|_aGVHD"), "[", 1)
    vireo_df$time_point <- sapply(vireo_df$cell, function(x) c("Baseline", "Tx", "D14", "D100", "GVHD")[sapply(c("_Baseline", "_Tx", "_D14", "_D100", "_aGVHD"), grepl, x)]) # Needs to match GVHD for GVHD
    vireo_df$sample_name <- paste0(vireo_df$patient_id, "_", vireo_df$time_point)

    
    
    # Add cell_id ident from Seurat object to vireo data 
    vireo_df <- dplyr::left_join(vireo_df, cell_id_ident, by="sample_name") %>% 
            dplyr::mutate(cell_id=paste0(cell_id, "_", cell_id_ident)) %>% 
            dplyr::select(-cell_id_ident)
                                  
    return(vireo_df)
                                  
}

In [12]:
vireo <- lapply(vireo_dir, vireo_import, cell_id_ident)
vireo <- do.call("rbind", vireo) %>% data.frame()

# Mutate vireo results

In [13]:
# Add cell type to vireo 
vireo <- dplyr::select(so@meta.data, cell_id, celltype_domain) %>% 
    dplyr::left_join(vireo, ., by="cell_id")

In [14]:
# Annotate genotype for samples by time point reference 
host_id_time_point <- vireo %>% 
    dplyr::filter(time_point %in% c("Baseline", "Tx")) %>% 
    dplyr::group_by(patient_id, donor_id) %>% 
    dplyr::summarise(n=n()) %>% 
    dplyr::filter(donor_id %in% c("donor0", "donor1")) %>% 
    dplyr::group_by(patient_id) %>% 
    slice_max(order_by=n, n=1, with_ties=FALSE) %>% ungroup() %>% 
    dplyr::rename(host_id=donor_id) %>% dplyr::select(-n)

# Annotate genotype for samples by cell type
host_id_cell_type <- vireo %>% 
    dplyr::filter(!vireo$patient_id %in% host_id_time_point$patient_id) %>% 
    dplyr::filter(time_point %in% c("GVHD")) %>% 
    dplyr::filter(celltype_domain %in% c("Non-hematopoetic")) %>% 
    dplyr::group_by(patient_id, donor_id) %>% 
    dplyr::summarise(n=n()) %>% 
    dplyr::filter(donor_id %in% c("donor0", "donor1")) %>% 
    dplyr::group_by(patient_id) %>% 
    slice_max(order_by=n, n=1, with_ties=FALSE) %>% ungroup() %>%
    dplyr::rename(host_id=donor_id) %>% dplyr::select(-n)

host_id <- rbind(host_id_time_point, host_id_cell_type)

`summarise()` has grouped output by 'patient_id'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'patient_id'. You can override using the
`.groups` argument.


In [15]:
# Add genotype to vireo result
vireo <- vireo %>% 
    dplyr::left_join(., host_id, by=c("patient_id")) %>%
    dplyr::mutate(genotype_class=ifelse(.$donor_id==.$host_id, "Host", ifelse(.$donor_id %in% c("donor0", "donor1"), "Donor", .$donor_id))) %>% 
    dplyr::mutate(genotype_best_class=ifelse(.$best_singlet==.$host_id, "Host", ifelse(.$best_singlet %in% c("donor0", "donor1"), "Donor", .$best_singlet)))

In [16]:
# XIST expression 
so[["pXIST_RNA"]] <- PercentageFeatureSet(so, features="XIST")
vireo <- dplyr::left_join(as.data.frame(vireo), so@meta.data[, c("cell_id", "pXIST_RNA")], by="cell_id")

# Check Host/Donor distribution 

In [ ]:
# Set factor levels 
set_factor_levels <- function(data) {
    
    data$genotype_class <- factor(data$genotype_class, levels=c("Host", "Donor", "unassigned", "doublet"))
    data$genotype_best_class <- factor(data$genotype_best_class, levels=c("Host", "Donor"))
    data$patient_id <- factor(data$patient_id, levels=paste0("Patient_", 1:12))
    data$time_point <- factor(data$time_point, levels=c("Baseline", "Tx", "D14", "D100", "GVHD"))
    
    return(data)
    
}

In [ ]:
vireo <- set_factor_levels(vireo)

In [ ]:
options(repr.plot.width=25, repr.plot.height=7.5)

plot_1 <-ggplot(vireo, aes(x=patient_id, fill=genotype_class)) + 
    geom_bar(position="fill") + 
    ggtitle("Genotype class") + xlab("") + ylab("ratio") + 
    scale_fill_manual("Vireo", values=setNames(c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425"), c("Host", "Donor", "unassigned", "doublet"))) + 
    facet_grid(celltype_domain~time_point) + 
    theme(axis.text.x=element_text(angle=45, vjust=1, hjust=1))

plot_1

In [ ]:
options(repr.plot.width=25, repr.plot.height=7.5)

plot_1 <-ggplot(vireo, aes(x=patient_id, fill=genotype_best_class)) + 
    geom_bar(position="fill") + 
    ggtitle("Genotype class") + xlab("") + ylab("ratio") + 
    scale_fill_manual("Vireo", values=setNames(c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425"), c("Host", "Donor", "unassigned", "doublet"))) + 
    facet_grid(celltype_domain~time_point) + 
    theme(axis.text.x=element_text(angle=45, vjust=1, hjust=1))

plot_1

# Set genotype class

In [ ]:
vireo$vireo_patient_id <- vireo$patient_id
vireo$vireo_donor_id <- vireo$donor_id
vireo$vireo_genotype_class <- vireo$genotype_class
vireo$vireo_genotype_best_class <- vireo$genotype_best_class

In [ ]:
vireo$genotype_class <- ifelse(vireo$time_point %in% c("Baseline", "Tx"), "Host", as.character(vireo$genotype_best_class))

In [ ]:
options(repr.plot.width=25, repr.plot.height=7.5)

plot_1 <-ggplot(vireo, aes(x=patient_id, fill=genotype_class)) + 
    geom_bar(position="fill") + 
    ggtitle("Genotype class") + xlab("") + ylab("ratio") + 
    scale_fill_manual("Vireo", values=setNames(c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425"), c("Host", "Donor", "unassigned", "doublet"))) + 
    facet_grid(celltype_domain~time_point) + 
    theme(axis.text.x=element_text(angle=45, vjust=1, hjust=1))

plot_1

# Save results

In [ ]:
# vireo[, c("cell_id", "vireo_patient_id", "vireo_donor_id", "vireo_genotype_best_class", "vireo_genotype_class", "genotype_class")] %>% tibble::column_to_rownames("cell_id") %>% write.csv("result/genotype/genotype.csv")